# Step 2B — LoRA Fine-Tuning + Inference

This notebook fine-tunes LLaMA 3 8B with QLoRA — one LoRA adapter per user — then
generates replies to each user's test triggers using their trained adapter.

**Workflow:**
1. Train an adapter for each user on their `train.csv` rows
2. Save each adapter to Google Drive
3. Load each adapter and generate replies for their `test.csv` triggers
4. Save results to `finetuned_results.csv`

**Before running:** Make sure you have run `preprocess.ipynb` first.

> ⚠️ Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)  
> ⏱ Expect ~15–30 min per user depending on dataset size.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install Dependencies

In [ ]:
!pip install -q transformers peft bitsandbytes datasets accelerate 'trl>=0.12.0' pandas torch

## 3. HuggingFace Login

You need access to the LLaMA 3 gated model:
1. Create a free account at [huggingface.co](https://huggingface.co)
2. Request access at [meta-llama/Meta-Llama-3-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)
3. Generate a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Configuration

**Edit these before running.**

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
TRAIN_PATH   = 'train.csv'
TEST_PATH    = 'test.csv'

# Adapters are saved here: ADAPTER_DIR/<user_id>/
ADAPTER_DIR  = 'adapters'

# Final results CSV
OUTPUT_PATH  = 'finetuned-results.csv'

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'

# ── LoRA hyperparameters ─────────────────────────────────────────────────────
# Rank: higher = more capacity but more VRAM. 16 is a good default.
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Training hyperparameters ─────────────────────────────────────────────────
NUM_EPOCHS     = 3
BATCH_SIZE     = 4
GRAD_ACCUM     = 2
LEARNING_RATE  = 2e-4

# ── Inference ────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 128

SEED = 42

## 5. Imports

In [ ]:
import os
import gc
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

print(f"GPU available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available : False


## 6. Load Train & Test Data

In [ ]:
def load_csv(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    return df

train_df = load_csv(TRAIN_PATH)
test_df  = load_csv(TEST_PATH)

users = ['U12', 'U14', 'U15']

print(f"Train rows : {len(train_df)}")
print(f"Test rows  : {len(test_df)}")
print(f"Users      : {users}")
print(f"\nTrain rows per user:")
print(train_df['user_id'].value_counts())

Train rows : 2109
Test rows  : 523
Users      : ['U12', 'U14', 'U15']

Train rows per user:
user_id
U15    1496
U09     176
U01     148
U12     112
U11      92
U14      55
U03      30
Name: count, dtype: int64


## 7. Helper Functions

In [ ]:
def build_system_prompt(user_id):
    """Persona prompt used during both training and inference — must be identical."""
    return (
        f"You are impersonating user '{user_id}'. "
        f"Reply to incoming text messages exactly as they would "
        f"matching their vocabulary, tone, punctuation, and message length. "
        f"Output only the reply, nothing else."
    )


def format_training_example(row, tokenizer, system_prompt):
    """
    Format one (incoming → reply) pair into the LLaMA 3 chat template.
    The model is trained to predict the reply given the incoming message.
    """
    messages = [
        {'role': 'system',    'content': system_prompt},
        {'role': 'user',      'content': row['incoming']},
        {'role': 'assistant', 'content': row['reply']},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}


def build_dataset(user_df, tokenizer, system_prompt):
    """Convert a user's training DataFrame into a HuggingFace Dataset."""
    records = [
        format_training_example(row, tokenizer, system_prompt)
        for _, row in user_df.iterrows()
    ]
    return Dataset.from_list(records)


def free_memory():
    """Release GPU memory between users."""
    gc.collect()
    torch.cuda.empty_cache()

## 8. Load Base Model (once)

The base model is loaded once and reused for all users.
Each user gets a fresh LoRA adapter layered on top.

> ⏱ First download is ~16GB — takes 5–10 min.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print('Loading base model in 4-bit (this may take a few minutes)...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
base_model = prepare_model_for_kbit_training(base_model)
print('✅ Base model loaded.')

Loading tokenizer...
Loading base model in 4-bit (this may take a few minutes)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## 9. Train One Adapter Per User

For each user:
- A fresh LoRA adapter is attached to the base model
- The adapter is trained on that user's training rows only
- The adapter weights (a few MB) are saved to Drive
- The adapter is detached so the next user starts clean

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
)

for user_id in users:
    print(f"\n{'='*55}")
    print(f"  Training adapter for: {user_id}")
    print(f"{'='*55}")

    # --- Build dataset ---
    user_train = train_df[train_df['user_id'] == user_id].reset_index(drop=True)
    print(f"  Training samples: {len(user_train)}")

    system_prompt = build_system_prompt(user_id)
    dataset = build_dataset(user_train, tokenizer, system_prompt)

    # --- Attach fresh LoRA adapter ---
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

    # --- Training config (SFTConfig replaces TrainingArguments in TRL >= 0.12) ---
    adapter_save_path = os.path.join(ADAPTER_DIR, str(user_id))
    os.makedirs(adapter_save_path, exist_ok=True)

    sft_config = SFTConfig(
        output_dir=adapter_save_path,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        bf16=True,
        fp16=False,
        logging_steps=10,
        save_strategy='epoch',
        save_total_limit=1,
        report_to='none',
        optim='paged_adamw_8bit',
        dataset_text_field='text'
    )

    # --- Train ---
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        args=sft_config,
    )
    trainer.train()

    # --- Save adapter weights only (small — a few MB) ---
    model.save_pretrained(adapter_save_path)
    tokenizer.save_pretrained(adapter_save_path)
    print(f"  ✅ Adapter saved to: {adapter_save_path}")

    # --- Detach adapter so base model is clean for next user ---
    base_model = model.get_base_model()
    del model, trainer
    free_memory()

print("\n✅ All adapters trained.")

## 10. Generate Replies — All Users

For each user:
- Load their saved LoRA adapter on top of the base model
- Use their `test.csv` rows as triggers (incoming messages)
- The `reply` column in test.csv is the held-out ground truth for evaluation

In [ ]:
users = ['U01', 'U03', 'U09', 'U11', 'U12', 'U14', 'U15']
all_results = []

for user_id in users:
    print(f"\n{'='*55}")
    print(f"  Inference for: {user_id}")
    print(f"{'='*55}")

    # Load adapter for this user
    adapter_path = os.path.join(ADAPTER_DIR, str(user_id))
    if not os.path.exists(adapter_path):
        print(f"  ⚠️  No adapter found at {adapter_path}, skipping.")
        continue

    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    system_prompt = build_system_prompt(user_id)

    # Get this user's test triggers
    user_test = test_df[test_df['user_id'] == user_id].reset_index(drop=True)
    print(f"  Triggers: {len(user_test)}\n")

    for i, row in user_test.iterrows():
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': row['incoming']},
        ]
        input_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors='pt'
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = output_ids[0][input_ids.shape[-1]:]
        generated  = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        all_results.append({
            'user_id'         : user_id,
            'incoming'        : row['incoming'],
            'ground_truth'    : row['reply'],
            'generated_reply' : generated,
        })

        print(f"  [{i+1}/{len(user_test)}] Incoming     : {row['incoming']}")
        print(f"           Ground truth : {row['reply']}")
        print(f"           Generated    : {generated}\n")

    # Free the adapter from memory before loading the next user's
    del model
    free_memory()

print("\n✅ All users complete.")

## 11. Save Results

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(results_df)} rows to: {OUTPUT_PATH}")
print(f"\nResults per user:")
print(results_df['user_id'].value_counts())
results_df.head(10)